# 02 — MCP

Build the provider's FastMCP server in-process and call each tool through `fastmcp.Client`.

No network. No A2A. Just MCP tool invocations against the closure-bound server.

Prereq: `anvil` is required only because `mint_credential` and `complete_swap` touch the chain — but in this notebook we'll only call the read-only tools.

## Setup

In [ ]:
import sys, pathlib
_ROOT = pathlib.Path.cwd().resolve()
if (_ROOT / 'shared').is_dir():
    sys.path.insert(0, str(_ROOT))
elif (_ROOT.parent / 'shared').is_dir():
    sys.path.insert(0, str(_ROOT.parent))


In [ ]:
from provider.mcp_server import build_mcp_server
from shared.config import Config
from fastmcp import Client
import json

PROVIDER = '0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d'
cfg = Config(provider_private_key=PROVIDER, sdn_mock=True)
mcp, tool_log = build_mcp_server(cfg)
_components = mcp._local_provider._components
_tools = {v.name: v for k, v in _components.items() if k.startswith('tool:')}
print('built provider MCP server with', len(_tools), 'tools')

## Run

List tools, then call the read-only ones.

In [ ]:
import asyncio

async def demo():
    async with Client(mcp) as c:
        tools = await c.list_tools()
        for t in tools:
            print('-', t.name)
        catalog = await c.call_tool('get_catalog', {})
        print('\nget_catalog →')
        for tier in json.loads(catalog.content[0].text):
            print(' ', tier)
        quote = await c.call_tool('request_quote',
            {'package_id': 'medium',
             'consumer_address': '0x000000000000000000000000000000000000dEaD'})
        print('\nrequest_quote → ', quote.content[0].text)
await demo()

## Inspect

In [ ]:
for entry in tool_log:
    print(entry)

## Teardown

(Nothing to do — MCP server is in-memory; Python GC reclaims it when the kernel ends.)